# Weeks 4–5 — Data Cleaning and Preparation

**Status:** Complete  
**Coverage:** January 2024 through September 2026  
**Confidentiality:** Aggregate results only; no MLS rows, addresses, coordinates, contact details, or identifiers are displayed.

This chapter documents how the mortgage-enriched Residential datasets become reproducible analysis-ready inputs for Week 6.

## 1. Objective and handbook checklist

The handbook requires us to:

- Convert `CloseDate`, `PurchaseContractDate`, `ListingContractDate`, and `ContractStatusChangeDate` to datetime values
- Remove unnecessary or redundant columns
- Handle missing values appropriately
- Confirm that analytical numeric fields are numeric
- Remove or flag nonpositive prices/areas, negative days on market, and negative bedroom/bathroom counts
- Create `listing_after_close_flag`, `purchase_after_close_flag`, and `negative_timeline_flag`
- Flag missing, zero, positive-longitude, and implausible coordinates
- Report before/after rows, data types, date flags, and geographic quality
- Save private analysis-ready CSVs and a reproducible `.py` script

## 2. Cleaning philosophy

Questionable records are **flagged rather than silently deleted**. This preserves the audit trail and lets each future metric use an explicit eligibility rule.

Only two types of columns are removed from the analysis-ready copies:

1. Fields that are completely empty in the dataset
2. Row-level contact/address fields that are not required for the handbook's later aggregate analysis

The original enriched CSVs remain unchanged. Optional missing values are retained as missing; no unsupported zero, mean, median, or text-placeholder imputation is used.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.weeks4_5_clean import (
    DATE_FIELDS,
    FLAG_COLUMNS,
    INPUT_FILES,
    OUTPUT_FILES,
    process_dataset,
)

pd.set_option("display.max_colwidth", 100)

## 3. Run the reproducible cleaning workflow

The next cell recreates both private outputs from the mortgage-enriched inputs. Console output is aggregate-only. It shows every removed field, parse confirmation, flag count, and output location.

In [ ]:
summaries = {
    dataset: process_dataset(dataset)
    for dataset in ("listings", "sold")
}

## 4. Before-and-after confirmation

No records were deleted. The net column count reflects 20 removals and 17 new quality flags in each dataset.

In [ ]:
coverage_table = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "input_rows": summary["input_rows"],
            "output_rows": summary["output_rows"],
            "input_columns": summary["input_columns"],
            "output_columns": summary["output_columns"],
            "rows_removed": summary["input_rows"] - summary["output_rows"],
        }
        for dataset, summary in summaries.items()
    ]
)
coverage_table

## 5. Missing values and removed columns

Eight fields are 100% missing in both datasets. Twelve row-level contact/address fields are also omitted from the analysis-ready copies. Office-level fields are retained because Week 6 explicitly requests competitive summaries by listing and buyer office.

In [ ]:
removed_columns = []
for dataset, summary in summaries.items():
    for column in summary["all_null_columns_removed"]:
        removed_columns.append(
            {
                "dataset": dataset,
                "column": column,
                "reason": "100% missing",
            }
        )
    for column in summary["unused_columns_removed"]:
        removed_columns.append(
            {
                "dataset": dataset,
                "column": column,
                "reason": "unnecessary row-level contact/address field",
            }
        )

pd.DataFrame(removed_columns)

## 6. Data type confirmation

The script validates dates as pandas datetime values before writing and confirms 21 analytical measures as numeric. CSV files do not store pandas dtypes, so downstream reads should parse the date fields or reuse the cleaning functions.

In [ ]:
date_type_table = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "column": column,
            "cleaning_dtype": dtype,
            "parse_failures": summary["date_parse_failures"][column],
        }
        for dataset, summary in summaries.items()
        for column, dtype in summary["date_dtypes"].items()
    ]
)

total_numeric_failures = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "numeric_fields_checked": len(summary["numeric_dtypes"]),
            "numeric_coercion_failures": sum(
                summary["numeric_coercion_failures"].values()
            ),
        }
        for dataset, summary in summaries.items()
    ]
)

display(date_type_table)
display(total_numeric_failures)

## 7. Date-consistency flags

The three required flags are defined as:

- `listing_after_close_flag`: listing date is later than close date
- `purchase_after_close_flag`: purchase-contract date is later than close date
- `negative_timeline_flag`: purchase-contract date is earlier than listing date

Comparisons are evaluated only when both relevant dates exist. A missing optional date is not automatically a chronology violation.

In [ ]:
def make_flag_table(flag_names):
    rows = []
    for dataset, summary in summaries.items():
        total = summary["output_rows"]
        for flag in flag_names:
            count = summary["flag_counts"][flag]
            rows.append(
                {
                    "dataset": dataset,
                    "flag": flag,
                    "flagged_records": count,
                    "share_percent": count / total * 100,
                }
            )
    return pd.DataFrame(rows)


date_flags = make_flag_table(
    [
        "date_parse_error_flag",
        "missing_required_date_flag",
        "listing_after_close_flag",
        "purchase_after_close_flag",
        "negative_timeline_flag",
    ]
)
date_flags

## 8. Invalid numeric flags

Missing numeric values are not automatically invalid. A flag is raised only when a present value violates the handbook rule. The combined `invalid_numeric_flag` makes later metric filtering transparent.

In [ ]:
numeric_flags = make_flag_table(
    [
        "numeric_parse_error_flag",
        "invalid_close_price_flag",
        "invalid_living_area_flag",
        "invalid_days_on_market_flag",
        "negative_bedrooms_flag",
        "negative_bathrooms_flag",
        "invalid_numeric_flag",
    ]
)
numeric_flags

## 9. Geographic data quality

The `implausible_coordinates_flag` uses a deliberately broad California screening rectangle: latitude 32.0–42.5 and longitude -125.0–-113.0. It is a review flag, not proof that a point is outside California. Precise school-district/state validation requires the Week 6 spatial-boundary join.

Flag categories can overlap. Use `invalid_coordinates_flag` for the combined number of affected records rather than adding the individual categories.

In [ ]:
geographic_flags = make_flag_table(
    [
        "missing_coordinates_flag",
        "zero_coordinates_flag",
        "positive_longitude_flag",
        "implausible_coordinates_flag",
        "invalid_coordinates_flag",
    ]
)
display(geographic_flags)

plot_data = geographic_flags.pivot(
    index="flag",
    columns="dataset",
    values="share_percent",
)
ax = plot_data.plot.bar(
    figsize=(10, 5),
    color=["#4C78A8", "#F28E2B"],
)
ax.set_title("Coordinate quality flags as a share of each dataset")
ax.set_xlabel("")
ax.set_ylabel("Flagged records (%)")
ax.legend(title="Dataset")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 10. Main findings

- All **655,943 listing rows** and **495,334 sold rows** were retained.
- All four date fields parsed with zero nonblank failures; all 21 selected measures converted with zero numeric failures.
- The required listing date was present for every listing record, and the required close date was present for every sold record.
- Date-order flags are uncommon: each individual flag affects less than 0.07% of either dataset.
- Required invalid-numeric conditions affect 459 listings (0.070%) and 234 sold records (0.047%).
- Missing coordinates dominate geographic quality: 81,364 listings (12.404%) and 4,797 sold records (0.968%).
- The combined coordinate flag affects 81,725 listings (12.459%) and 4,912 sold records (0.992%).
- The original enriched files remain unchanged, and the cleaned outputs contain no all-null or intentionally removed contact/address fields.

## 11. Decisions and supervisor questions

### Decisions

- Flag invalid records instead of deleting them.
- Do not invent replacements for optional missing values.
- Retain listing identifiers for auditability and future grain decisions.
- Retain office fields for Week 6 segment analysis.
- Treat the California rectangle as a coarse screen only.

### Questions for the supervisor

1. For Week 6 volume metrics, should the counting unit be rows, unique `ListingKey`, unique property, or completed transaction?
2. Should records with any date-quality flag be excluded from duration metrics, or shown as a separate quality category?
3. Should records missing coordinates remain eligible for non-geographic market metrics?
4. Is the rectangular coordinate screen sufficient until the school-district spatial join, or should we add a California boundary polygon now?

## 12. Deliverables and next step

Weeks 4–5 deliverables:

- `src/weeks4_5_clean.py`
- `data/processed/analysis_ready_listings_residential.csv` (private)
- `data/processed/analysis_ready_sold_residential.csv` (private)
- `reports/weeks4_5_cleaning_report.md`
- This notebook

Week 6 will calculate close-to-original-list ratio, price per square foot, year/month variables, listing-to-contract days, and contract-to-close days. Each metric should document which Weeks 4–5 flags define an eligible record.